<img src="https://res.cloudinary.com/dtizipxds/image/upload/q_auto/f_auto/v1776264397/banner_yvwajv.png" width="100%">


In [ ]:
%pip install numpy pandas matplotlib seaborn mlxtend


# Solution - Association Rules


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

sns.set_theme(style="whitegrid")
df = pd.read_csv(Path("data.csv"))
print(df.shape)


In [ ]:
transactions = df.groupby("transaction_id")["itemDescription"].apply(list).tolist()
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
basket = pd.DataFrame(te_ary, columns=te.columns_)

ap_itemsets = apriori(basket, min_support=0.015, use_colnames=True)
fp_itemsets = fpgrowth(basket, min_support=0.015, use_colnames=True)

ap_rules = association_rules(ap_itemsets, metric="confidence", min_threshold=0.15)
fp_rules = association_rules(fp_itemsets, metric="confidence", min_threshold=0.15)

ap_rules = ap_rules.sort_values(["lift", "confidence"], ascending=False)
fp_rules = fp_rules.sort_values(["lift", "confidence"], ascending=False)
print("Apriori rules:", len(ap_rules), "FP-Growth rules:", len(fp_rules))


In [ ]:
def build_tidsets(basket_df):
    tidsets = {}
    for col in basket_df.columns:
        idx = set(basket_df.index[basket_df[col]].tolist())
        if idx:
            tidsets[frozenset([col])] = idx
    return tidsets


def eclat_recursive(prefix, items, min_support_count, out):
    items = sorted(items, key=lambda x: len(x[1]), reverse=True)
    while items:
        item, tidset = items.pop(0)
        new_itemset = prefix.union(item)
        support_count = len(tidset)
        if support_count >= min_support_count:
            out.append((new_itemset, support_count))
            suffix = []
            for other_item, other_tidset in items:
                inter = tidset.intersection(other_tidset)
                if len(inter) >= min_support_count:
                    suffix.append((other_item, inter))
            if suffix:
                eclat_recursive(new_itemset, suffix, min_support_count, out)


def eclat(basket_df, min_support=0.015):
    min_count = int(np.ceil(min_support * len(basket_df)))
    base = list(build_tidsets(basket_df).items())
    out = []
    eclat_recursive(frozenset(), base, min_count, out)
    rows = []
    for itemset, count in out:
        rows.append({"itemsets": itemset, "support": count / len(basket_df), "support_count": count})
    return pd.DataFrame(rows).sort_values("support", ascending=False)

eclat_itemsets = eclat(basket)
print("ECLAT itemsets:", len(eclat_itemsets))
eclat_itemsets.head(10)


In [ ]:
plot_df = ap_rules.head(150).copy()
plot_df["rule_size"] = plot_df["antecedents"].apply(len) + plot_df["consequents"].apply(len)

plt.figure(figsize=(8, 5))
sns.scatterplot(data=plot_df, x="support", y="confidence", size="lift", hue="rule_size", alpha=0.8)
plt.title("Top Apriori Rules")
plt.tight_layout()
plt.show()

final_rules = ap_rules[["antecedents", "consequents", "support", "confidence", "lift"]].head(150).copy()
final_rules["antecedents"] = final_rules["antecedents"].apply(lambda x: ', '.join(sorted(list(x))))
final_rules["consequents"] = final_rules["consequents"].apply(lambda x: ', '.join(sorted(list(x))))
final_rules.to_csv("solution.csv", index=False)
final_rules.head()
